In [63]:
import joblib
import pandas as pd
import numpy as np


 Load processed_data & trained model

In [64]:
df=pd.read_csv('processed_telco_churn.csv')
rf= joblib.load("customer_churn_model.pkl")
X=df.drop('Churn',axis=1)

Predict churn

In [65]:
df["predicted_churn"] = rf.predict(X)


 Predict churn probability

In [66]:
df["Churn_Probability"] = rf.predict_proba(X)[:, 1]

Create Customer Grade

In [67]:

def customer_grade(prob):

    if prob >= 0.90:
        return "Grade A (Critical Risk)"

    elif prob >= 0.75:
        return "Grade B (High Risk)"

    elif prob >= 0.50:
        return "Grade C (Medium Risk)"

    elif prob >= 0.30:
        return "Grade D (Low Risk)"

    else:
        return "Grade E (Safe)"

Calculate CLV, Priority Score

In [68]:
#Calculate CLV
df["CLV"]=df["MonthlyCharges"]*df["tenure"]
# Priority Score (0–100)
df["Priority_Score"] = (
    df["Churn_Probability"] * 0.7 +
    (df["CLV"] / df["CLV"].max()) * 0.3
) * 100

df["Priority_Score"] = df["Priority_Score"].round(2)

Assign Priority Level to each customer

In [69]:
def priority_level(score):
    if score >= 80:
        return "Critical"
    elif score >= 60:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"

df["Priority_Level"] = df["Priority_Score"].apply(priority_level)


Create Smart Retention Recommendation

In [70]:
def smart_retention(row):

    priority = row["Priority_Level"]
    clv = row["CLV"]
    tenure = row["tenure"]
    monthly = row["MonthlyCharges"]

    # Critical Priority
    if priority == "Critical":

        if clv >= 5000:
            return "Immediate Call + 30% Discount + Dedicated Relationship Manager"

        elif tenure < 12:
            return "Free 2-Month Subscription + Onboarding Support"

        elif monthly >= 80:
            return "25% Discount + Premium Support"

        else:
            return "20% Discount + Loyalty Rewards"

    # High Priority
    elif priority == "High":

        if clv >= 5000:
            return "Premium Support + Service Upgrade"

        elif monthly >= 80:
            return "20% Discount"

        else:
            return "15% Discount + Loyalty Points"

    # Medium Priority
    elif priority == "Medium":

        if tenure < 12:
            return "Welcome Offer + Personalized Email"

        else:
            return "10% Discount + Plan Upgrade Recommendation"

    # Low Priority
    else:

        if clv >= 5000:
            return "Loyalty Rewards + Thank You Coupon"

        else:
            return "Regular Promotional Email"

df["Retention_Action"] = df.apply(smart_retention, axis=1)     

In [71]:

retention_report = df[[
    "Churn_Probability",
    "CLV",
    "Priority_Score",
    "Priority_Level",
    "Retention_Action"
]].sort_values(
    by="Priority_Score",
    ascending=False
)

retention_report.head(20)

,Churn_Probability,CLV,Priority_Score,Priority_Level,Retention_Action
3950,0.800584,4713.80,72.58,High,20% Discount
4960,0.795138,4785.00,72.45,High,20% Discount
3681,0.800579,4646.00,72.34,High,20% Discount
6453,0.796687,4702.35,72.27,High,20% Discount
6032,0.670491,7160.40,72.06,High,Premium Support + Service Upgrade
4074,0.737815,5724.60,71.73,High,Premium Support + Service Upgrade
3330,0.769991,4987.80,71.40,High,20% Discount
6009,0.744011,5504.05,71.39,High,Premium Support + Service Upgrade
2948,0.813427,4091.80,71.30,High,20% Discount
4502,0.750757,5324.00,71.23,High,Premium Support + Service Upgrade


In [72]:
df.to_csv("customer_retention_result.csv", index=False)

print("Retention results saved successfully!")

Retention results saved successfully!
